# 04 - Loading the data into MySQL and creating reporting views

So far, every clean table has lived only as a CSV in
`datasets/processed/`. In this notebook I load them into MySQL, so
Power BI can read from one organized place instead of a bunch of
loose CSV files.

Why use a database? Because with SQL I can **join** tables and run
queries that a single CSV can't answer -- for example, linking
production to quality through the same work order.


In [1]:
import os
import glob
import pandas as pd
from sqlalchemy import create_engine, text

PROCESSED = "../../datasets/processed"
DIM = "../../datasets/dim"
SQL_TABLE = "../sql_table"
SQL_VIEW = "../sql_view"


## Step 1: connection details

I read the username and password from **environment variables** (or a
`.env` file), never hard-coded in the notebook -- if this notebook goes
on GitHub, the password shouldn't go with it.

Before running this notebook, set the following (in a terminal, or in
a `.env` file at the project root):

```
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=your_password_here
MYSQL_DB=manufacturing_performance_analytics
```


In [2]:
try:
    from dotenv import load_dotenv
    load_dotenv("../../.env")
except ImportError:
    print("python-dotenv is not installed -- run: pip install python-dotenv")

user = os.environ.get("MYSQL_USER", "root")
password = os.environ.get("MYSQL_PASSWORD", "")
host = os.environ.get("MYSQL_HOST", "localhost")
port = os.environ.get("MYSQL_PORT", "3306")
database = os.environ.get("MYSQL_DB", "manufacturing_performance_analytics")

print(f"Connecting to: {user}@{host}:{port}/{database}")


Connecting to: root@localhost:3306/manufacturing_performance_analytics


## Step 2: create the database (if it doesn't exist yet)

In [3]:
# connect to the server WITHOUT specifying a database yet (it may not exist)
server_connection = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}")

with server_connection.begin() as connection:
    connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {database} CHARACTER SET utf8mb4"))

print(f"Database '{database}' ready.")

# now a connection pointed straight at the database. local_infile=1 is needed
# for Step 4 below, which loads the CSVs with LOAD DATA LOCAL INFILE instead
# of row-by-row INSERTs (see the note in Step 4 for why)
engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4&local_infile=1"
)

# the server also needs local_infile turned on (it's off by default). this
# tries to turn it on for the current session; if the account doesn't have
# permission, LOAD DATA will fail later with a clear "not allowed" error --
# in that case, add local_infile=1 under [mysqld] in my.ini and restart MySQL
try:
    with engine.begin() as connection:
        connection.execute(text("SET GLOBAL local_infile = 1"))
    print("local_infile enabled on the server.")
except Exception as e:
    print(f"Could not set local_infile automatically ({e}). "
          "If Step 4 fails, enable local_infile=1 in my.ini and restart MySQL.")


Database 'manufacturing_performance_analytics' ready.
local_infile enabled on the server.


## Step 3: create the tables (schema)

The `01_create_fact_tables.sql` and `02_create_dim_tables.sql` files
already have the `CREATE TABLE` statement for each table, with the
right column types. `run_sql_file()` reads the file, strips out the
comment-header lines, splits what's left on `;` (each statement ends
with one), and runs each statement.

**Bug fixed here:** the previous version only checked whether each
statement (after splitting on `;`) started with `--`, but every one of
these `.sql` files opens with a multi-line comment header that has no
`;` in it -- so that header stayed glued to the first real statement,
and since the *combined* text started with `--`, the whole thing was
silently thrown away. In practice this meant the very first table (or
view, in Step 5) in every `.sql` file was never actually created --
here that was `fact_production_processed`, in `02_create_dim_tables.sql`
it was `dim_masterbatch`, and one of the four process views in Step 5
was missing the same way. Stripping comment *lines* before splitting
on `;`, instead of checking whole statements, fixes it for every file.


In [4]:
def run_sql_file(engine, file_path):
    with open(file_path, encoding="utf-8") as f:
        full_sql = f.read()

    # strip out full-line comments first (lines starting with --), *before*
    # splitting on ";". Every .sql file here opens with a comment header
    # with no ";" in it, so without this step the header stays glued to
    # the first real statement -- and since the combined text starts with
    # "--", the old version of this function (which only checked whether
    # each already-split chunk started with "--") silently discarded that
    # entire first statement, header and all. That's how the very first
    # CREATE TABLE / CREATE VIEW in every one of these files was quietly
    # never being run.
    sql_without_comments = "\n".join(
        line for line in full_sql.splitlines() if not line.strip().startswith("--")
    )
    statements = [s.strip() for s in sql_without_comments.split(";") if s.strip()]

    with engine.begin() as connection:
        for statement in statements:
            connection.execute(text(statement))

    print(f"{len(statements)} statements executed from {file_path}")


run_sql_file(engine, f"{SQL_TABLE}/01_create_fact_tables.sql")
run_sql_file(engine, f"{SQL_TABLE}/02_create_dim_tables.sql")


29 statements executed from ../sql_table/01_create_fact_tables.sql


14 statements executed from ../sql_table/02_create_dim_tables.sql


## Step 4: load the data

Every CSV file in `datasets/processed/` becomes a table with the same
name (minus the `.csv`). I use that rule so I'm not typing out the
name of each of the 34 tables by hand -- I just loop over the files in
the folder.

Before loading, I empty the table first (`TRUNCATE`), so this notebook
can be run again without duplicating any rows.

**Why not `pandas.to_sql()`?** I used that first, but two of these
tables have 80k-125k rows, and `to_sql()` builds and sends a Python
`INSERT` statement for every chunk of rows -- for those two tables
alone it took over a minute each, which is why this notebook felt like
it was stuck on the loop over files.

Instead, `load_csv_into_table()` below uses **`LOAD DATA LOCAL
INFILE`**: MySQL reads and parses the CSV itself, server-side, instead
of Python building the INSERT statements. Same result, but the two
biggest tables now load in about 2 seconds each instead of over a
minute -- roughly a 30x speed-up (I benchmarked both versions before
switching). The one thing this needs some manual handling for: `LOAD
DATA` doesn't know that an empty cell should become `NULL`, or that
the text "True"/"False" pandas writes for boolean columns should
become `1`/`0`, and a couple of rows can carry a stray `\r` if a file
ever gets regenerated with `\n`-only line endings instead of Windows'
`\r\n` -- so the function loads each column into a MySQL user variable
first and cleans it up with a `CASE` expression before it's written to
the real column.


In [5]:
import time

def load_csv_into_table(engine, csv_path, table_name):
    """
    Bulk-loads a CSV straight into a MySQL table with LOAD DATA LOCAL
    INFILE, instead of pandas.to_sql(). See the note in Step 4 above
    for why -- short version: MySQL parses the file itself, so there's
    no per-chunk INSERT statement being built and sent from Python.
    """
    # LOAD DATA needs an absolute path with forward slashes -- a Windows
    # backslash path would get misread as an escape sequence inside the SQL string
    csv_path_sql = os.path.abspath(csv_path).replace(os.sep, "/")

    # only reads the header row (not the whole file) to get the column names,
    # in the same order as they appear in the CSV
    columns = pd.read_csv(csv_path, encoding="utf-8-sig", nrows=0).columns.tolist()

    # each CSV column is first loaded into a MySQL user variable (@c0, @c1, ...)
    # instead of going straight into the table column, so it can pass through
    # the CASE below before being written to the real column:
    #  - TRIM(TRAILING '\r' ...) -- lines are split on '\n' only (below), so
    #    on a CRLF file the last column of every row would otherwise keep a
    #    stray '\r'. Windows (pandas' default here) writes CRLF, but this
    #    also makes the loader work unchanged if a CSV is ever regenerated
    #    with LF-only line endings.
    #  - '' -> NULL, since an empty CSV cell should mean "no value", not
    #    the literal text ''
    #  - 'True'/'False' -> 1/0, because that's the text pandas writes for
    #    boolean columns, and LOAD DATA doesn't convert that on its own
    user_vars = ", ".join(f"@c{i}" for i in range(len(columns)))
    set_clause = ",\n            ".join(
        f"`{col}` = CASE WHEN TRIM(TRAILING '\r' FROM @c{i}) = '' THEN NULL "
        f"WHEN TRIM(TRAILING '\r' FROM @c{i}) = 'True' THEN 1 "
        f"WHEN TRIM(TRAILING '\r' FROM @c{i}) = 'False' THEN 0 "
        f"ELSE TRIM(TRAILING '\r' FROM @c{i}) END"
        for i, col in enumerate(columns)
    )

    load_sql = f"""
        LOAD DATA LOCAL INFILE '{csv_path_sql}'
        INTO TABLE `{table_name}`
        CHARACTER SET utf8mb4
        FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
        LINES TERMINATED BY '\n'
        IGNORE 1 LINES
        ({user_vars})
        SET {set_clause}
    """

    with engine.begin() as connection:
        connection.execute(text(f"TRUNCATE TABLE {table_name}"))

    t0 = time.time()
    # LOAD DATA isn't exposed through SQLAlchemy's text()/execute() the normal
    # way, so this drops down to the underlying pymysql connection to run it
    raw_connection = engine.raw_connection()
    try:
        cursor = raw_connection.cursor()
        cursor.execute(load_sql)
        rows_loaded = cursor.rowcount
        raw_connection.commit()
    finally:
        raw_connection.close()

    print(f"loaded: {table_name} ({rows_loaded:,} rows, {time.time() - t0:.1f}s)")


# 1) every fact/dimension table already sitting in datasets/processed
for path in sorted(glob.glob(f"{PROCESSED}/*.csv")):
    table_name = os.path.basename(path).replace(".csv", "")
    load_csv_into_table(engine, path, table_name)

# 2) the dimension tables that come straight from datasets/dim (these
#    don't go through the cleaning notebooks, they arrive ready from engineering)
dim_files_loaded_directly = [
    "dim_masterbatch.csv",
    "dim_machine_setup.csv",
    "dim_cap_control_plan_cq.csv",
    "dim_bottle_control_plan_cq.csv",
    "dim_ink_control_plan_cq.csv",
]
for file_name in dim_files_loaded_directly:
    path = f"{DIM}/{file_name}"
    table_name = file_name.replace(".csv", "")
    load_csv_into_table(engine, path, table_name)


loaded: dim_bottle (101 rows, 0.0s)
loaded: dim_cap (31 rows, 0.0s)
loaded: dim_customer (14 rows, 0.0s)
loaded: dim_ink (99 rows, 0.0s)
loaded: dim_machine (18 rows, 0.0s)
loaded: dim_mold (215 rows, 0.0s)
loaded: dim_operator (18 rows, 0.0s)
loaded: dim_raw_material_control_plan (41 rows, 0.0s)
loaded: dim_supplier (8 rows, 0.0s)


loaded: fact_bottle_attribute_inspection_cq_processed (17,520 rows, 0.3s)
loaded: fact_bottle_disposition_lot_cq_processed (2,190 rows, 0.1s)


loaded: fact_bottle_inspection_variables_cq_processed (124,580 rows, 2.9s)
loaded: fact_cap_attribute_inspection_cq_processed (9,924 rows, 0.2s)


loaded: fact_cap_disposition_lot_cq_processed (1,654 rows, 0.0s)


loaded: fact_cap_inspection_variable_cq_processed (83,947 rows, 2.4s)
loaded: fact_capa_processed (357 rows, 0.0s)
loaded: fact_customer_complaints_processed (397 rows, 0.0s)


loaded: fact_downtime_processed (21,809 rows, 0.5s)
loaded: fact_ink_attribute_inspection_cq_processed (4,456 rows, 0.1s)
loaded: fact_ink_disposition_lot_cq_processed (557 rows, 0.0s)


loaded: fact_material_consumption_processed (23,434 rows, 0.2s)
loaded: fact_nonconformance_processed (512 rows, 0.0s)


loaded: fact_production_plan_processed (9,110 rows, 0.2s)


loaded: fact_production_processed (9,110 rows, 0.2s)
loaded: fact_raw_material_inspection_processed (4,446 rows, 0.1s)
loaded: fact_raw_material_lot_disposition_processed (1,193 rows, 0.0s)


loaded: fact_sales_processed (7,088 rows, 0.1s)
loaded: fact_supplier_complaints_processed (38 rows, 0.0s)
loaded: ml_predictions_downtime_forecast (4 rows, 0.0s)
loaded: ml_predictions_downtime_forecast_history (41 rows, 0.0s)
loaded: ml_predictions_lot_quality (751 rows, 0.0s)
loaded: ml_predictions_predictive_maintenance (18 rows, 0.0s)
loaded: ml_predictions_predictive_maintenance_history (1,311 rows, 0.0s)
loaded: ml_predictions_production_forecast (4 rows, 0.0s)
loaded: ml_predictions_production_forecast_history (41 rows, 0.0s)
loaded: ml_predictions_rejected_forecast (4 rows, 0.0s)
loaded: ml_predictions_rejected_forecast_history (41 rows, 0.0s)
loaded: ml_predictions_scrap_rate (1,789 rows, 0.0s)
loaded: dim_masterbatch (132 rows, 0.0s)
loaded: dim_machine_setup (38 rows, 0.0s)
loaded: dim_cap_control_plan_cq (10 rows, 0.0s)
loaded: dim_bottle_control_plan_cq (13 rows, 0.0s)
loaded: dim_ink_control_plan_cq (8 rows, 0.0s)


## Step 5: create the views (rolling 52-week window)

A **view** is a saved SQL query that behaves like a table. The four
views here always filter to the most recent 52 weeks of data -- that's
what Power BI uses, not the full tables (otherwise the dashboard would
show the entire accumulated history, with no rolling window).


In [6]:
view_files = [
    "01_production_by_process_52w.sql",
    "02_downtime_material_plan_52w.sql",
    "03_quality_control_52w.sql",
    "04_quality_assurance_52w.sql",
    "05_dim_views.sql",
]

for file_name in view_files:
    run_sql_file(engine, f"{SQL_VIEW}/{file_name}")

print("Views created. Power BI can now connect (tables starting with vw_).")


4 statements executed from ../sql_view/01_production_by_process_52w.sql
5 statements executed from ../sql_view/02_downtime_material_plan_52w.sql


8 statements executed from ../sql_view/03_quality_control_52w.sql
7 statements executed from ../sql_view/04_quality_assurance_52w.sql
14 statements executed from ../sql_view/05_dim_views.sql
Views created. Power BI can now connect (tables starting with vw_).
